# SkinFLNet++ — ISIC2018 / ISIC2019 cross evaluation

You need **checkpoints** under `results/isic2019/<run_name>/` and `results/isic2018/<run_name>/`. Use the **download** cell below to fetch ISIC2019/2018 challenge files into `DATA_ROOT`, or place data there yourself before building manifests.

**What this notebook does**

1. **ISIC2019 model → ISIC2018 test:** loads the 8-class global checkpoint, evaluates on ISIC2018 `split=test`, and scores **7-way** metrics using only logits `[:, :7]` (SCC logit dropped at inference). Class indices **0–6** match both datasets (slot 3 is AK on 2019 vs AKIEC on 2018 in this repo’s encoding).
2. **ISIC2018 model → ISIC2019 test:** loads the 7-class global checkpoint, **drops all test rows with label 7 (SCC)**, then evaluates on the remaining ISIC2019 test images with **7-way** metrics.

**Order**

Setup → (optional) install → **download ISIC2019 + ISIC2018** (official challenge zips/CSVs) → **build manifests** for both years → configuration → run cross-eval.

**Data note:** ISIC2018 uses the official **train / val / test** folders (including `ISIC2018_Task3_Test_*`). ISIC2019 manifests in this repo are built from **Training** metadata + images; `split=test` is an **internal stratified holdout** from training (not the challenge `ISIC_2019_Test_Input` labels). The download still includes official 2019 test archives for completeness.

**Colab:** same `SKINFL_RUNTIME` / Drive pattern as the other SkinFL notebooks; `DATA_ROOT` stays on session disk for I/O.


In [ ]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"This project requires Python >= 3.11 (pyproject.toml). Got: {sys.version}"
    )


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False


def _runtime_mode() -> str:
    env = os.environ.get("SKINFL_RUNTIME", "").strip().lower()
    if env in ("local", "colab_drive"):
        return env
    return "colab_drive" if _in_colab() else "local"


COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive/SkinFL")
COLAB_SESSION_DATA_ROOT = Path("/content/skinfl_data")
PROJECT_ROOT_OVERRIDE: Path | None = None
DROP_MISSING_IMAGES: bool | None = None


def _discover_project_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    for cand in candidates:
        if (cand / "pyproject.toml").is_file() and (cand / "src").is_dir():
            return cand
    for parent in start.parents:
        if (parent / "pyproject.toml").is_file() and (parent / "src").is_dir():
            return parent
    raise FileNotFoundError(
        "Could not find SkinFL repo (pyproject.toml + src/). "
        "cd to repo root or notebooks/, or set PROJECT_ROOT_OVERRIDE, "
        "or SKINFL_RUNTIME=colab_drive on Colab."
    )


RUNTIME = _runtime_mode()

if RUNTIME == "colab_drive":
    try:
        from google.colab import drive
    except ImportError as e:
        raise SystemExit(
            "SKINFL_RUNTIME=colab_drive but not in Colab. "
            "Unset SKINFL_RUNTIME or use local Jupyter."
        ) from e
    drive.mount("/content/drive")
    PROJECT_ROOT = COLAB_DRIVE_ROOT
else:
    PROJECT_ROOT = PROJECT_ROOT_OVERRIDE or _discover_project_root()

if RUNTIME == "colab_drive":
    DATA_ROOT = COLAB_SESSION_DATA_ROOT
else:
    DATA_ROOT = PROJECT_ROOT / "data"

RESULTS_ISIC2019 = PROJECT_ROOT / "results" / "isic2019"
RESULTS_ISIC2018 = PROJECT_ROOT / "results" / "isic2018"
OUT_CROSS = PROJECT_ROOT / "results" / "cross_eval"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
for path in (RESULTS_ISIC2019, RESULTS_ISIC2018, OUT_CROSS):
    path.mkdir(parents=True, exist_ok=True)

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(f"Invalid PROJECT_ROOT: {PROJECT_ROOT} (pyproject.toml missing).")

os.chdir(PROJECT_ROOT)
_repo_root = str(PROJECT_ROOT.resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

_drop = (RUNTIME == "colab_drive") if DROP_MISSING_IMAGES is None else DROP_MISSING_IMAGES
if _drop:
    os.environ["SKINFL_DROP_MISSING_IMAGES"] = "1"
else:
    os.environ.pop("SKINFL_DROP_MISSING_IMAGES", None)

print("RUNTIME:", RUNTIME)
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ISIC2019:", RESULTS_ISIC2019)
print("RESULTS_ISIC2018:", RESULTS_ISIC2018)
print("OUT_CROSS:", OUT_CROSS)
print(
    "SKINFL_DROP_MISSING_IMAGES:",
    os.environ.get("SKINFL_DROP_MISSING_IMAGES", "(unset — strict image checks)"),
)


In [ ]:
# %pip install -q -e ".[dev]"


## Download ISIC2019 + ISIC2018 (official challenge files)

Fetches curated ISIC S3 URLs into `DATA_ROOT/ISIC2019/` and `DATA_ROOT/ISIC2018/` (large downloads; use Colab GPU runtime + session disk).

- **Colab (`colab_drive`):** downloads run automatically when layouts are missing (unless `FORCE_ISIC_REDOWNLOAD = False` and data already looks complete).
- **Local:** set `ALLOW_ISIC_DOWNLOAD_ON_LOCAL = True` in the next cell to run the same downloads; otherwise the cell skips and you must place data under `DATA_ROOT` yourself.

After this cell, run **Build manifests** so `manifest.csv` exists for both datasets.

In [ ]:
# ISIC2019 + ISIC2018: official challenge files -> DATA_ROOT (large; VM session disk on Colab).
# Includes ISIC2018 Task3 Test_Input + Test_GroundTruth and ISIC2019 Test_Input + test CSVs.

import shutil
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

# Local Jupyter: set True to run the same URL downloads (large disk + time).
ALLOW_ISIC_DOWNLOAD_ON_LOCAL = False

# True = always re-fetch and re-extract (slow).
FORCE_ISIC_REDOWNLOAD = False


def _download_file(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = Request(url, headers={"User-Agent": "SkinFL-ISIC-fetch/1.0"})
    with urlopen(req) as resp:  # noqa: S310 — curated ISIC S3 URLs only
        hdr = resp.headers.get("Content-Length")
        total = int(hdr) if hdr is not None and hdr.isdigit() else None
        bs = 256 * 1024
        if tqdm is None:
            with open(dest, "wb") as f:
                while True:
                    chunk = resp.read(bs)
                    if not chunk:
                        break
                    f.write(chunk)
            return
        with open(dest, "wb") as f, tqdm(
            desc=dest.name[:48],
            total=total,
            unit="iB",
            unit_scale=True,
            unit_divisor=1024,
            miniters=1,
        ) as bar:
            while True:
                chunk = resp.read(bs)
                if not chunk:
                    break
                f.write(chunk)
                bar.update(len(chunk))


def _isic2019_training_ready() -> bool:
    root = DATA_ROOT / "ISIC2019"
    gt = root / "ISIC_2019_Training_GroundTruth.csv"
    meta = root / "ISIC_2019_Training_Metadata.csv"
    img_dir = root / "ISIC_2019_Training_Input"
    if not gt.is_file() or not meta.is_file() or not img_dir.is_dir():
        return False
    return any(img_dir.glob("*.jpg"))


def _download_isic2019_bundle() -> None:
    urls = [
        "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip",
        "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip",
        "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv",
        "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv",
        "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Metadata.csv",
        "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv",
    ]
    if _isic2019_training_ready() and not FORCE_ISIC_REDOWNLOAD:
        print(
            "SKIP ISIC2019: training CSVs + Training_Input already under",
            DATA_ROOT / "ISIC2019",
        )
        return
    isic_dest = DATA_ROOT / "ISIC2019"
    staging = DATA_ROOT / ".cache_isic2019"
    isic_dest.mkdir(parents=True, exist_ok=True)
    staging.mkdir(parents=True, exist_ok=True)
    for url in urls:
        name = Path(url.rstrip("/")).name
        out = staging / name
        _download_file(url, out)
        if name.endswith(".csv"):
            shutil.copy2(out, isic_dest / name)
            print("  -> copied CSV to", isic_dest / name)
        elif name.endswith(".zip"):
            print("  -> extracting:", name, flush=True)
            with zipfile.ZipFile(out, "r") as zf:
                infos = zf.infolist()
                if tqdm is not None:
                    for m in tqdm(infos, desc=f"Unpack {name[:42]}", unit="file"):
                        zf.extract(m, isic_dest)
                else:
                    zf.extractall(isic_dest)
    train_in = isic_dest / "ISIC_2019_Training_Input"
    if not train_in.is_dir():
        raise FileNotFoundError(
            f"After extract, expected directory missing: {train_in} (check ZIP layout)."
        )
    for p in staging.iterdir():
        p.unlink(missing_ok=True)
    print("Done ISIC2019. Training_Input:", train_in.resolve())


def _extract_csv_from_zip(zip_path: Path, dest_csv: Path) -> None:
    with zipfile.ZipFile(zip_path, "r") as zf:
        csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
        if not csv_members:
            raise FileNotFoundError(f"No CSV in {zip_path}")
        with zf.open(csv_members[0]) as src, open(dest_csv, "wb") as out:
            shutil.copyfileobj(src, out)


def _copy_repo_fallback_csvs(isic_dest: Path) -> None:
    root = PROJECT_ROOT
    pairs = [
        (root / "ISIC2018_Task3_Training_GroundTruth.csv", isic_dest / "ISIC2018_Task3_Training_GroundTruth.csv"),
        (root / "ISIC2018_Task3_Training_LesionGroupings (1).csv", isic_dest / "ISIC2018_Task3_Training_LesionGroupings.csv"),
        (root / "ISIC2018_Task3_Training_LesionGroupings.csv", isic_dest / "ISIC2018_Task3_Training_LesionGroupings.csv"),
    ]
    for src_p, dst_p in pairs:
        if src_p.is_file() and not dst_p.is_file():
            shutil.copy2(src_p, dst_p)
            print("  -> copied fallback", src_p.name)


def _isic2018_ready() -> bool:
    root = DATA_ROOT / "ISIC2018"
    train_in = root / "ISIC2018_Task3_Training_Input"
    csvs = [
        root / "ISIC2018_Task3_Training_GroundTruth.csv",
        root / "ISIC2018_Task3_Validation_GroundTruth.csv",
        root / "ISIC2018_Task3_Test_GroundTruth.csv",
        root / "ISIC2018_Task3_Training_LesionGroupings.csv",
    ]
    if not train_in.is_dir() or not all(p.is_file() for p in csvs):
        return False
    return any(train_in.glob("*.jpg"))


def _download_isic2018_bundle() -> None:
    urls = [
        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Training_Input.zip",
        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Training_GroundTruth.zip",
        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Training_LesionGroupings.csv",
        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Validation_Input.zip",
        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Validation_GroundTruth.zip",
        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Test_Input.zip",
        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Test_GroundTruth.zip",
    ]
    isic_dest = DATA_ROOT / "ISIC2018"
    isic_dest.mkdir(parents=True, exist_ok=True)
    if _isic2018_ready() and not FORCE_ISIC_REDOWNLOAD:
        print("SKIP ISIC2018: already present under", DATA_ROOT / "ISIC2018")
        _copy_repo_fallback_csvs(isic_dest)
        return
    staging = DATA_ROOT / ".cache_isic2018"
    staging.mkdir(parents=True, exist_ok=True)
    for url in urls:
        name = Path(url.rstrip("/")).name
        out = staging / name
        _download_file(url, out)
        if name.endswith(".csv"):
            shutil.copy2(out, isic_dest / name)
            print("  -> copied CSV to", isic_dest / name)
        elif name.endswith(".zip"):
            print("  -> extracting:", name, flush=True)
            if "GroundTruth" in name:
                csv_name = name.replace(".zip", ".csv")
                _extract_csv_from_zip(out, isic_dest / csv_name)
                print("  -> extracted CSV to", isic_dest / csv_name)
            else:
                with zipfile.ZipFile(out, "r") as zf:
                    infos = zf.infolist()
                    if tqdm is not None:
                        for m in tqdm(infos, desc=f"Unpack {name[:42]}", unit="file"):
                            zf.extract(m, isic_dest)
                    else:
                        zf.extractall(isic_dest)
    train_in = isic_dest / "ISIC2018_Task3_Training_Input"
    if not train_in.is_dir():
        raise FileNotFoundError(f"Missing after extract: {train_in}")
    _copy_repo_fallback_csvs(isic_dest)
    for p in staging.iterdir():
        p.unlink(missing_ok=True)
    print("Done ISIC2018. Training_Input:", train_in.resolve())


_should_download = (RUNTIME == "colab_drive") or ALLOW_ISIC_DOWNLOAD_ON_LOCAL
if not _should_download:
    print(
        "SKIP ISIC downloads: use Colab (colab_drive) or set ALLOW_ISIC_DOWNLOAD_ON_LOCAL = True. "
        "Copying optional repo-root ISIC2018 CSV fallbacks if present."
    )
    _copy_repo_fallback_csvs(DATA_ROOT / "ISIC2018")
else:
    _download_isic2019_bundle()
    _download_isic2018_bundle()


In [ ]:
# Build manifest.csv for ISIC2019 and ISIC2018 (run after the download cell, or ensure data is under DATA_ROOT).
from src.data.manifest_build import ensure_manifest_for_dataset

_root = Path(DATA_ROOT)
ensure_manifest_for_dataset(_root, "isic2019")
ensure_manifest_for_dataset(_root, "isic2018")


## Configuration

Edit ``RUN_NAME_*`` to match your completed FL runs (folder names under ``results/isic2019`` / ``results/isic2018``). Checkpoints may be ``checkpoint_latest.pt`` or ``model_round_NNN.pth`` inside that folder.


In [ ]:
from pathlib import Path

# Federated run directories (must contain checkpoint_latest.pt and/or model_round_*.pth)
RUN_NAME_2019 = "isic2019_dirichlet"
RUN_NAME_2018 = "isic2018_dirichlet"

CKPT_2019 = RESULTS_ISIC2019 / RUN_NAME_2019
CKPT_2018 = RESULTS_ISIC2018 / RUN_NAME_2018

BACKBONE = "vgg16_bn"
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
SEED = 42

from src.eval.isic_cross_eval import SHARED_SEVEN_CLASS_NAMES, resolve_checkpoint_file_in_run_dir


def _print_checkpoint(label: str, path: Path) -> Path:
    p = Path(path)
    resolved = resolve_checkpoint_file_in_run_dir(p) if p.is_dir() else p
    print(f"{label}: {resolved} (exists={resolved.is_file()})")
    return resolved


print("Aligned 7-class indices (0–6):", SHARED_SEVEN_CLASS_NAMES)
_print_checkpoint("ISIC2019", CKPT_2019)
_print_checkpoint("ISIC2018", CKPT_2018)


## Run cross-evaluation

Writes a JSON summary under ``OUT_CROSS`` (metrics + metadata). Requires GPU for reasonable speed if datasets are large.


In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import torch

from src.eval.isic_cross_eval import (
    eval_2018_model_on_2019_test_no_scc,
    eval_2019_model_on_2018_test,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "device": str(device),
    "backbone": BACKBONE,
    "isic2019_on_isic2018": None,
    "isic2018_on_isic2019_no_scc": None,
}

summary["isic2019_on_isic2018"] = eval_2019_model_on_2018_test(
    data_root=DATA_ROOT,
    checkpoint_path=CKPT_2019,
    backbone=BACKBONE,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    device=device,
)
print("2019 → 2018 test | macro_f1:", summary["isic2019_on_isic2018"]["metrics"].get("macro_f1"))

summary["isic2018_on_isic2019_no_scc"] = eval_2018_model_on_2019_test_no_scc(
    data_root=DATA_ROOT,
    checkpoint_path=CKPT_2018,
    backbone=BACKBONE,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    device=device,
)
print("2018 → 2019 test (no SCC) | macro_f1:", summary["isic2018_on_isic2019_no_scc"]["metrics"].get("macro_f1"))

out_path = OUT_CROSS / "cross_eval_summary.json"
# JSON-serializable: convert float nan if any (metrics are plain floats from sklearn path)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)
print("Wrote", out_path.resolve())
